In [1]:
!pip install -q accelerate==0.34.2 peft==0.12.0 bitsandbytes==0.43.3 transformers==4.43.1 trl==0.10.1 sacrebleu evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [19]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    EarlyStoppingCallback
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd
import numpy as np
import evaluate

In [3]:
from google.colab import userdata
sec_key = userdata.get('HF_TOKEN')

In [4]:
!huggingface-cli login --token $sec_key

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [5]:
# The model that you want to train from the Hugging Face hub
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [6]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [29]:
pipeline = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

In [18]:
messages = [
    {"role": "user", "content": """
    A partir de ahora hablas español. Escoge solo la respuesta correcta:
    En cuál de las reacciones siguientes participa la biotina.
     a) Hidroxilaciones.
     b) Carboxilaciones.
     c) Deshidrataciones.
     d) Desaminaciones.
    """},
]

outputs = pipeline(
    messages,
    max_new_tokens=20,
)
print(outputs[0]["generated_text"][-1].get('content'))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


La respuesta correcta es: b) Carboxilaciones.

La biotina es un cof


In [16]:
messages = [
    {"role": "user", "content": """
    ¿A qué variable de personalidad alude la definición “tendencia a experimentar emociones negativas junto con una elevada inhibición social”?
    a) Depresión.
    b) Personalidad Tipo D.
    c) Indefensión.
    d) Patrón de conducta Tipo A.
    e) Hostilidad.
    """},
]

outputs = pipeline(
    messages,
    max_new_tokens=20,
)
print(outputs[0]["generated_text"][-1].get('content'))

La respuesta correcta es:

b) Personalidad Tipo D.

La Personalidad Tipo D se caracter


In [17]:
messages = [
    {"role": "user", "content": """
    A partir de ahora hablas español. Escoge solo la respuesta correcta:
    Adolescente de 16 años que consulta por amenorrea primaria. No posee antecedentes medicoquirúrgicos de interés. A la exploración física encontramos caracteres sexuales secundarios femeninos, genitales externos femeninos, con un desarrollo mamario estadio III-IV, escaso vello púbico y axilar. Presenta una estatura en el percentil 90 para su edad. La radiografía simple revela una edad ósea de 15.8 años. El estudio analítico revela un estradiol de 50 pg/ml, y unas gonadotropinas elevadas. ¿Cuál de las siguientes entidades clínicas sería compatible con el diagnóstico?
    a) Síndrome de Klinefelter.
    b) Síndrome de Kallman.
    c) Retraso constitucional del crecimiento.
    d) Hiperplasia suprarrenal congénita.
    e) Síndrome de insensibilidad androgénica completo.
    """},
]

outputs = pipeline(
    messages,
    max_new_tokens=20,
)
print(outputs[0]["generated_text"][-1].get('content'))

La respuesta correcta es:

b) Síndrome de Kallman.

La descripción


In [20]:
# The instruction dataset to use
dataset_name = "head_qa"
# Load dataset (you can process it here)
dataset_train = load_dataset(dataset_name, split="train", trust_remote_code=True)
dataset_test = load_dataset(dataset_name, split="test")
dataset_validation = load_dataset(dataset_name, split="validation")
dataset_train[2]

head_qa.py:   0%|          | 0.00/6.21k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

head-qa-es-en-pdfs.zip:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2657 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2742 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1366 [00:00<?, ? examples/s]

{'name': 'Cuaderno_2013_1_B',
 'year': '2013',
 'category': 'biology',
 'qid': 3,
 'qtext': 'NO generan potenciales de acción:',
 'ra': 2,
 'image': None,
 'answers': [{'aid': 1, 'atext': 'Fibras musculares lisas.'},
  {'aid': 2, 'atext': 'Neuronas bipolares de la retina.'},
  {'aid': 3, 'atext': 'Fibras musculares estriadas esqueléticas.'},
  {'aid': 4, 'atext': 'Fibras musculares cardíacas.'},
  {'aid': 5, 'atext': 'Neuronas ganglionares de la retina.'}]}

In [21]:
def flatten_dataset(dataset):
    data_df = pd.DataFrame(dataset)
    answers = pd.json_normalize(data_df['answers'])
    df_answers = pd.DataFrame()
    data_df['answer_text'] = None
    df_answers['pos_answer'] = None
    data_df['ra'] = data_df['ra'].astype(int)
    for c in range(len(answers.columns)):
        answer = pd.json_normalize(answers[c])
        df_answers = pd.concat([df_answers, answer['atext']], axis=1)
        df_answers.rename(columns={'atext': c + 1}, inplace=True)
    for i in range(data_df.shape[0]):
        col = data_df.loc[i, 'ra']
        data_df.loc[i, 'answer_text'] = df_answers.iloc[i, col]
    df_answers['pos_answer'] = df_answers.apply(lambda x: '\n '.join(x.dropna().astype(str)), axis=1)
    data_df = pd.concat([data_df, df_answers], axis=1)
    return data_df

In [22]:
data_test_df = flatten_dataset(dataset_test)

In [23]:
preguntas = []
for i in range(data_test_df.shape[0]):
    preguntas.append(data_test_df.loc[i, 'qtext'] + "\n" + data_test_df.loc[i, 'pos_answer'])
preguntas[0:10]

['Forma fibras extracelulares con gran resistencia a la tensión:\nFibronectina.\n Colágeno.\n Integrinas.\n Proteoglucanos.',
 'El fosfolípido cardiolipina es abundante en la membrana:\nMitocondrial interna.\n Mitocondrial externa.\n Plasmática.\n Lisosomal.',
 'NO es una función de los filamentos intermedios:\nProporcionar apoyo estructural a la célula.\n Fijar el núcleo en su sitio.\n Proveer resistencia mecánica.\n Locomoción celular.',
 'Los cuerpos multivesiculares son:\nPeroxisomas.\n Mitocondrias.\n Polisomas.\n Endosomas.',
 'Forman la envuelta de mielina de los axones en el sistema nervioso periférico:\nOligodendrocitos.\n Células de Schwann.\n Células microgliales.\n Células ganglionares.',
 'El fibrocartílago se localiza en:\nLaringe.\n Extremos ventrales de costillas.\n Discos intervertebrales.\n Epiglotis.',
 'La molécula lineal de ADN asociada a proteínas es:\nNucleosoma.\n Cromatina.\n Cromosoma eucariótico.\n Gen.',
 'Son células multinucleadas:\nCondroblastos.\n Osteob

In [35]:
pred_resp = []

for pregunta in preguntas[0:10]:
    messages = [{"role": "user", "content": pregunta}]
    outputs = pipeline(
        messages,
        max_new_tokens=20,
    )
    pred_resp.append(outputs[0]["generated_text"][-1].get('content'))
pred_resp[0:10]

KeyboardInterrupt: 

In [40]:
referencias = []
for i in range(data_test_df.shape[0]):
    referencias.append(data_test_df.loc[i, 'answer_text'])
referencias[0:10]

['Colágeno.',
 'Mitocondrial interna.',
 'Locomoción celular.',
 'Endosomas.',
 'Células de Schwann.',
 'Discos intervertebrales.',
 'Cromosoma eucariótico.',
 'Osteoclastos.',
 'Cartílago.',
 'Endocondral.']

In [41]:
len(pred_resp)

122

In [44]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=77d74759a61efb3ddd6541d4f6625a50797ec036635ead2912e21ab56677c808
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [46]:
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=pred_resp,
                         references=referencias[0:len(pred_resp)],
                        use_aggregator=True)
print(list(results.keys()))
print(results["rouge1"])
print(results["rouge2"])
print(results["rougeL"])

['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
0.2226356198230403
0.12688355838092744
0.21626798141744538


In [47]:
sacrebleu = evaluate.load("sacrebleu")

In [48]:
results = sacrebleu.compute(predictions=pred_resp,
                             references=referencias[0:len(pred_resp)])
print(list(results.keys()))
print(round(results["score"], 1))

['score', 'counts', 'totals', 'precisions', 'bp', 'sys_len', 'ref_len']
7.8
